# Candidate retrieval, one idea at a time

This notebook does **not** train a model. It answers one question: given the products in one session, which other products are reasonable possibilities?

We use one simple rule: if products appeared together in many past sessions, they are related.

In [1]:
from pathlib import Path
import sys

import polars as pl

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval.covisitation import build_covisitation_matrix

For learning, we use 10,000 complete sessions from the random development sample. A session is never cut in half.

In [2]:
events = pl.read_parquet(PROJECT_ROOT / "data/processed/train_dev.parquet")
demo_sessions = events.select("session").unique().sort("session").head(10_000)
demo_events = events.join(demo_sessions, on="session", how="inner")

print(f"Sessions: {demo_events.select('session').n_unique():,}")
print(f"Events: {len(demo_events):,}")

Sessions: 10,000
Events: 391,581


`aid_x -> aid_y` means that `aid_y` appeared in the same session as `aid_x`. The score is the number of different sessions that produced that pair.

In [3]:
general_matrix = build_covisitation_matrix(demo_events, weighting="general", top_k=10)
general_matrix.head(10)

aid_x,aid_y,score
i64,i64,f64
1,4541,1.0
1,28092,1.0
1,190430,1.0
1,375012,1.0
1,504137,1.0
1,526193,1.0
1,675264,1.0
1,757471,1.0
1,930325,1.0


Now choose one session. Its history is what a real recommender would know when making a prediction.

In [4]:
session_id = demo_events.select("session").unique().sort("session").item(0, 0)
history = demo_events.filter(pl.col("session") == session_id).sort("ts", descending=True)
history

session,aid,ts,type
i64,i64,i64,str
36,357864,1661672467026,"""clicks"""
36,1624059,1661672461532,"""clicks"""
36,357864,1661672450084,"""clicks"""
36,195036,1661671851280,"""clicks"""
36,357864,1661671755054,"""clicks"""
…,…,…,…
36,166037,1659323813974,"""clicks"""
36,1847645,1659323733584,"""clicks"""
36,1240180,1659323663292,"""clicks"""


Each product in the history votes for its related products. If the same product is related to several history items, we add those votes together.

In [5]:
history_aids = history.get_column("aid").unique().to_list()

related_items = (
    general_matrix
    .filter(pl.col("aid_x").is_in(history_aids))
    .group_by("aid_y")
    .agg(pl.col("score").sum().alias("related_score"))
    .sort("related_score", descending=True)
    .head(20)
)

related_items

aid_y,related_score
i64,f64
357864,15.0
98928,15.0
205516,15.0
301441,15.0
195036,15.0
…,…
762779,2.0
231487,2.0
1505419,2.0


The final candidate list keeps both kinds of products: items already in the session and newly retrieved related items. A later model will score this small list instead of scoring every product in the catalog.

In [6]:
candidate_list = (
    pl.concat([
        history.select(pl.col("aid").alias("candidate")).unique(),
        related_items.select(pl.col("aid_y").alias("candidate")),
    ])
    .unique()
    .with_columns(pl.col("candidate").is_in(history_aids).alias("already_in_session"))
    .join(related_items.rename({"aid_y": "candidate"}), on="candidate", how="left")
    .with_columns(pl.col("related_score").fill_null(0))
    .sort(["related_score", "already_in_session"], descending=[True, True])
)

candidate_list

candidate,already_in_session,related_score
i64,bool,f64
98928,true,15.0
357864,true,15.0
301441,true,15.0
205516,true,15.0
195036,true,15.0
…,…,…
1409076,true,0.0
177909,true,0.0
58161,true,0.0


You have now completed the retrieval half of a recommender:

`session history -> related products -> candidate list`

The next notebook can explain how we tell whether these candidates contain the hidden future product.